In [ ]:
from datasets import load_dataset

ds = load_dataset('yahma/alpaca-cleaned', split='train')
ds = ds.shuffle(seed = 42).select(range(1500))

In [19]:
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 1500
})

In [20]:
def to_prompt_completion(ex):
    q = ex['instruction']

    if ex['input']:
        q += "n\n" + ex['input']
    
    return {
            "promt":[{'role': 'user', "content": q}],
            "completion": [{'role': "assistent", "content": ex['output']}]
            }


In [21]:
ds = ds.map(to_prompt_completion, remove_columns=ds.column_names)
ds = ds.train_test_split(test_size=150, seed = 42)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [22]:
ds

DatasetDict({
    train: Dataset({
        features: ['promt', 'completion'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['promt', 'completion'],
        num_rows: 150
    })
})

In [1]:
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
import torch 

model = AutoModelForCausalLM.from_pretrained("Qwen3-4B-Base", torch_dtype = torch.float32)

cfg = SFTConfig(
    output_dir='my_llm_sft_v1',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=10,
    learning_rate=1e-5,
    lr_scheduler_type= 'cosine',
    num_train_epochs=3,
    warmup_ratio=0.2,
    bf16=True,
    gradient_checkpointing=True,
    max_length=2048,
    completion_only_loss=True,
    eval_strategy="steps",
    eval_steps= 50
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

: 

In [ ]:
trainer = SFTTrainer(model=model, args=cfg, processing_class)